In [2]:

from pathlib import Path
import pandas as pd

project_dir = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")

files_to_check = [
    project_dir / "data" / "raw" / "NexziByte_Final_Scored.csv",
    project_dir / "data" / "clean" / "Day3_Prospects_Clean.csv",
]

for file_path in files_to_check:
    print("\n" + "=" * 70)
    print("FILE:", file_path)
    print("EXISTS:", file_path.exists())

    if file_path.exists():
        data = pd.read_csv(file_path)
        print("SHAPE:", data.shape)
        print("COLUMNS:")
        for column in data.columns:
            print("-", column)

        quote_keywords = [
            "quote", "quotation", "proposal", "win", "loss",
            "accepted", "price", "discount", "revision",
            "outcome", "response", "meeting", "interaction"
        ]

        relevant_columns = [
            column for column in data.columns
            if any(word in column.lower() for word in quote_keywords)
        ]

        print("\nPOTENTIALLY RELEVANT COLUMNS:")
        print(relevant_columns)



FILE: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\raw\NexziByte_Final_Scored.csv
EXISTS: True
SHAPE: (180, 25)
COLUMNS:
- Company Name
- Website / Domain
- Industry / Sub-industry
- HQ Location (City, State)
- Founded Year (Standardized)
- Products/Services
- Target Customers / B2B or B2C
- Technology Signals
- Recent Business Signal
- Decision-Maker (Name + Title)
- Source(s) Used
- Date Verified
- Employee Count (Numeric)
- Website Validation
- Employee Validation
- Duplicate Flag
- Data Quality Status
- Company Size Score
- B2B Fit Score
- Technology Signal Score
- Business Need Score
- Recent Business Signal Score
- Decision-Maker Score
- Prospect Score
- Prospect Classification

POTENTIALLY RELEVANT COLUMNS:
[]

FILE: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\clean\Day3_Prospects_Clean.csv
EXISTS: True
SHAPE: (180, 28)
COLUMNS:
- Company Name
- Website / Domain
- Industry / Sub-industry
- HQ Location (City, State)
- Founded Year (Standardized)
- Products/Services


In [3]:

# Day 7 - Quotation Data Readiness Audit

from pathlib import Path
import pandas as pd

project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

raw_path = project_dir / "data" / "raw" / "NexziByte_Final_Scored.csv"
clean_path = project_dir / "data" / "clean" / "Day3_Prospects_Clean.csv"

audit_rows = [
    {
        "Dataset": "Original prospect dataset",
        "Path": str(raw_path),
        "Records": 180,
        "Quotation records available": "No",
        "Verified win/loss outcome available": "No",
        "Model readiness": "Not ready for real quote-win modeling"
    },
    {
        "Dataset": "Cleaned prospect dataset",
        "Path": str(clean_path),
        "Records": 180,
        "Quotation records available": "No",
        "Verified win/loss outcome available": "No",
        "Model readiness": "Not ready for real quote-win modeling"
    }
]

audit_df = pd.DataFrame(audit_rows)

output_path = (
    project_dir / "data" / "processed"
    / "Day7_Quote_Data_Readiness_Audit.csv"
)

audit_df.to_csv(output_path, index=False)

print(audit_df.to_string(index=False))
print("\nAudit saved to:", output_path)
print("Rows saved:", len(audit_df))


                  Dataset                                                                              Path  Records Quotation records available Verified win/loss outcome available                       Model readiness
Original prospect dataset D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\raw\NexziByte_Final_Scored.csv      180                          No                                  No Not ready for real quote-win modeling
 Cleaned prospect dataset D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\clean\Day3_Prospects_Clean.csv      180                          No                                  No Not ready for real quote-win modeling

Audit saved to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day7_Quote_Data_Readiness_Audit.csv
Rows saved: 2


In [4]:

# Day 7 - Required Quotation Data Catalogue

required_quote_fields = [
    ["Quote ID", "Unique quotation identifier", "Required"],
    ["Company ID", "Company linked to the quotation", "Required"],
    ["Quote Date", "Date the quotation was issued", "Required"],
    ["Quote Value", "Quoted monetary value", "Required"],
    ["Discount", "Discount offered", "If available"],
    ["Revision Count", "Number of quotation revisions", "If available"],
    ["Engagement History", "Meetings and interactions before quote", "If available"],
    ["Quote Status", "Accepted, lost, pending, or other defined status", "Required"],
    ["Accepted Date", "Date accepted, where applicable", "If available"],
    ["Loss Reason", "Recorded reason for a lost quotation", "If available"],
]

quote_fields_df = pd.DataFrame(
    required_quote_fields,
    columns=["Field", "Business Purpose", "Requirement"]
)

output_path = (
    project_dir / "data" / "processed"
    / "Day7_Required_Quotation_Fields.csv"
)

quote_fields_df.to_csv(output_path, index=False)

print(quote_fields_df.to_string(index=False))
print("\nSaved to:", output_path)
print("Fields documented:", len(quote_fields_df))


             Field                                 Business Purpose  Requirement
          Quote ID                      Unique quotation identifier     Required
        Company ID                  Company linked to the quotation     Required
        Quote Date                    Date the quotation was issued     Required
       Quote Value                            Quoted monetary value     Required
          Discount                                 Discount offered If available
    Revision Count                    Number of quotation revisions If available
Engagement History           Meetings and interactions before quote If available
      Quote Status Accepted, lost, pending, or other defined status     Required
     Accepted Date                  Date accepted, where applicable If available
       Loss Reason             Recorded reason for a lost quotation If available

Saved to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day7_Required_Quotation_Fields.csv
Fi

In [5]:

# Day 7 - Quotation Win Target Definition

target_rules = [
    {
        "Status": "Won",
        "Proposed Definition": "Quotation accepted with a recorded acceptance date",
        "Training Treatment": "Include after business validation",
        "Validation Status": "Pending business validation"
    },
    {
        "Status": "Lost",
        "Proposed Definition": "Quotation explicitly recorded as lost",
        "Training Treatment": "Include after business validation",
        "Validation Status": "Pending business validation"
    },
    {
        "Status": "Pending",
        "Proposed Definition": "Quotation remains open or awaits a decision",
        "Training Treatment": "Exclude from binary win/loss training",
        "Validation Status": "Pending business validation"
    },
    {
        "Status": "Unknown",
        "Proposed Definition": "Status or supporting evidence is insufficient",
        "Training Treatment": "Flag for review; exclude from training",
        "Validation Status": "Pending business validation"
    }
]

target_df = pd.DataFrame(target_rules)

output_path = (
    project_dir / "data" / "processed"
    / "Day7_Quote_Target_Definition.csv"
)

target_df.to_csv(output_path, index=False)

print(target_df.to_string(index=False))
print("\nSaved to:", output_path)
print("Target rules saved:", len(target_df))


 Status                                Proposed Definition                     Training Treatment           Validation Status
    Won Quotation accepted with a recorded acceptance date      Include after business validation Pending business validation
   Lost              Quotation explicitly recorded as lost      Include after business validation Pending business validation
Pending        Quotation remains open or awaits a decision  Exclude from binary win/loss training Pending business validation
Unknown      Status or supporting evidence is insufficient Flag for review; exclude from training Pending business validation

Saved to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day7_Quote_Target_Definition.csv
Target rules saved: 4


In [6]:
# Day 7 - Quote Model Limitations

limitations = [
    {
        "Area": "Quotation records",
        "Finding": "No quotation-level records found in the available raw or cleaned prospect datasets.",
        "Impact": "Cannot calculate actual quotation win probabilities."
    },
    {
        "Area": "Win/loss outcomes",
        "Finding": "No verified quotation acceptance or loss outcomes are available.",
        "Impact": "Cannot train or evaluate a real quote-win model."
    },
    {
        "Area": "Quotation features",
        "Finding": "Quote value, discount, revisions, engagement history, and loss reasons are unavailable.",
        "Impact": "Cannot analyze actual win/loss drivers."
    },
    {
        "Area": "Target definition",
        "Finding": "The proposed win/loss definitions require business validation, including the 90-day acceptance window.",
        "Impact": "Final target rules require confirmation before production modeling."
    },
    {
        "Area": "Recommended next step",
        "Finding": "Request historical quotation records with verified outcomes from the business owner.",
        "Impact": "Use verified data for production modeling; keep any synthetic demonstration clearly labeled."
    }
]

limitations_df = pd.DataFrame(limitations)

output_path = (
    project_dir / "data" / "processed"
    / "Day7_Quote_Model_Limitations.csv"
)

limitations_df.to_csv(output_path, index=False)

print(limitations_df.to_string(index=False))
print("\nSaved to:", output_path)
print("Limitations documented:", len(limitations_df))

                 Area                                                                                                Finding                                                                                       Impact
    Quotation records                    No quotation-level records found in the available raw or cleaned prospect datasets.                                         Cannot calculate actual quotation win probabilities.
    Win/loss outcomes                                       No verified quotation acceptance or loss outcomes are available.                                             Cannot train or evaluate a real quote-win model.
   Quotation features                Quote value, discount, revisions, engagement history, and loss reasons are unavailable.                                                      Cannot analyze actual win/loss drivers.
    Target definition The proposed win/loss definitions require business validation, including the 90-day acceptance window.    

In [7]:

# Day 7 - Step 6: Generate Synthetic Quotation Dataset
# DEMONSTRATION ONLY — NOT REAL NEXZIBYTE BUSINESS DATA

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n_quotes = 300

quote_values = rng.integers(15000, 500001, size=n_quotes)
discount_pct = np.round(rng.uniform(0, 25, size=n_quotes), 2)
revision_count = rng.integers(0, 5, size=n_quotes)
engagement_count = rng.integers(0, 11, size=n_quotes)

# Synthetic probability mechanism for demonstration only.
logit = (
    -1.0
    + 0.10 * engagement_count
    + 0.07 * discount_pct
    - 0.35 * revision_count
)

win_probability = 1 / (1 + np.exp(-logit))
is_won = rng.binomial(1, win_probability)

# Generate some pending quotations separately.
is_pending = rng.random(n_quotes) < 0.15

outcomes = []
for i in range(n_quotes):
    if is_pending[i]:
        outcomes.append("Pending")
    elif is_won[i] == 1:
        outcomes.append("Won")
    else:
        outcomes.append("Lost")

quote_dates = (
    pd.Timestamp("2026-01-01")
    + pd.to_timedelta(
        rng.integers(0, 240, size=n_quotes), unit="D"
    )
)

accepted_dates = []
loss_reasons = []

for i, outcome in enumerate(outcomes):
    if outcome == "Won":
        accepted_dates.append(
            quote_dates[i] + pd.Timedelta(
                days=int(rng.integers(1, 91))
            )
        )
        loss_reasons.append("")
    elif outcome == "Lost":
        accepted_dates.append(pd.NaT)
        loss_reasons.append(
            rng.choice([
                "Price",
                "Competitor",
                "Timing",
                "Requirements mismatch"
            ])
        )
    else:
        accepted_dates.append(pd.NaT)
        loss_reasons.append("")

synthetic_quotes = pd.DataFrame({
    "Quote_ID": [f"SYN-Q-{i:04d}" for i in range(1, n_quotes + 1)],
    "Company_ID": [f"SYN-C-{i:04d}" for i in range(1, n_quotes + 1)],
    "Quote_Date": quote_dates,
    "Quote_Value": quote_values,
    "Discount_Pct": discount_pct,
    "Revision_Count": revision_count,
    "Engagement_Count": engagement_count,
    "Quote_Status": outcomes,
    "Accepted_Date": accepted_dates,
    "Loss_Reason": loss_reasons,
    "Synthetic_Data_Flag": True,
    "Data_Use": "Demonstration only; not real business data"
})

output_path = (
    project_dir / "data" / "synthetic"
    / "Day7_Synthetic_Quotation_Dataset.csv"
)

synthetic_quotes.to_csv(output_path, index=False)

print("Dataset shape:", synthetic_quotes.shape)
print("\nQuotation status counts:")
print(synthetic_quotes["Quote_Status"].value_counts())
print("\nSynthetic data flag:")
print(synthetic_quotes["Synthetic_Data_Flag"].all())
print("\nSaved to:", output_path)


Dataset shape: (300, 12)

Quotation status counts:
Quote_Status
Lost       137
Won        118
Pending     45
Name: count, dtype: int64

Synthetic data flag:
True

Saved to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day7_Synthetic_Quotation_Dataset.csv


C:\Users\Sai\AppData\Local\Temp\ipykernel_33548\3948522451.py:51: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  quote_dates[i] + pd.Timedelta(


In [17]:

# Day 7 - Step 7: Prepare Synthetic Modeling Dataset
# DEMONSTRATION ONLY — NOT REAL BUSINESS PERFORMANCE

model_df = synthetic_quotes[
    synthetic_quotes["Quote_Status"].isin(["Won", "Lost"])
].copy()

# Binary target: Won = 1, Lost = 0
model_df["Won_Flag"] = (
    model_df["Quote_Status"] == "Won"
).astype(int)

# Select model features and target
feature_columns = [
    "Quote_Value",
    "Discount_Pct",
    "Revision_Count",
    "Engagement_Count"
]

X = model_df[feature_columns].copy()
y = model_df["Won_Flag"].copy()

# Verify dataset
print("Modeling dataset shape:", model_df.shape)
print("\nOutcome counts:")
print(model_df["Quote_Status"].value_counts())

print("\nTarget distribution:")
print(y.value_counts().rename(index={0: "Lost (0)", 1: "Won (1)"}))

print("\nFeature columns:", feature_columns)
print("Missing values in features:", int(X.isna().sum().sum()))
print("Missing target values:", int(y.isna().sum()))
print("Pending records included:", int((model_df["Quote_Status"] == "Pending").sum()))
print("All included records are synthetic:",
      bool(model_df["Synthetic_Data_Flag"].all()))

# Save the prepared dataset
output_path = (
    project_dir / "data" / "synthetic"
    / "Day7_Synthetic_Quote_Modeling_Dataset.csv"
)

model_df.to_csv(output_path, index=False)

print("\nSaved to:", output_path)


Modeling dataset shape: (255, 13)

Outcome counts:
Quote_Status
Lost    137
Won     118
Name: count, dtype: int64

Target distribution:
Won_Flag
Lost (0)    137
Won (1)     118
Name: count, dtype: int64

Feature columns: ['Quote_Value', 'Discount_Pct', 'Revision_Count', 'Engagement_Count']
Missing values in features: 0
Missing target values: 0
Pending records included: 0
All included records are synthetic: True

Saved to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day7_Synthetic_Quote_Modeling_Dataset.csv


In [18]:

# Day 7 - Step 8: Train/Test Split
# SYNTHETIC DEMONSTRATION ONLY

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts().rename(index={0: "Lost (0)", 1: "Won (1)"}))

print("\nTesting target distribution:")
print(y_test.value_counts().rename(index={0: "Lost (0)", 1: "Won (1)"}))

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Total rows:", len(X_train) + len(X_test))
print("Any overlap between train and test indices:",
      not set(X_train.index).isdisjoint(set(X_test.index)))


Training feature shape: (204, 4)
Testing feature shape: (51, 4)

Training target distribution:
Won_Flag
Lost (0)    110
Won (1)      94
Name: count, dtype: int64

Testing target distribution:
Won_Flag
Lost (0)    27
Won (1)     24
Name: count, dtype: int64

Training rows: 204
Testing rows: 51
Total rows: 255
Any overlap between train and test indices: False


In [19]:

# Day 7 - Step 9: Train Baseline Quote Win Model
# SYNTHETIC DEMONSTRATION ONLY — NOT A REAL BUSINESS MODEL

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

quote_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42, max_iter=1000))
])

# Fit only on training data
quote_model.fit(X_train, y_train)

# Predict on held-out test data
y_pred = quote_model.predict(X_test)
y_prob = quote_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
metrics = {
    "Model": "Logistic Regression",
    "Dataset": "Synthetic demonstration only",
    "Test Records": len(y_test),
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1 Score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

print("Baseline Model Evaluation")
print("-" * 35)

for name, value in metrics.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Lost", "Won"],
    zero_division=0
))


Baseline Model Evaluation
-----------------------------------
Model: Logistic Regression
Dataset: Synthetic demonstration only
Test Records: 51
Accuracy: 0.5882
Precision: 0.5714
Recall: 0.5000
F1 Score: 0.5333
ROC-AUC: 0.6620

Confusion Matrix:
[[18  9]
 [12 12]]

Classification Report:
              precision    recall  f1-score   support

        Lost       0.60      0.67      0.63        27
         Won       0.57      0.50      0.53        24

    accuracy                           0.59        51
   macro avg       0.59      0.58      0.58        51
weighted avg       0.59      0.59      0.59        51



In [20]:

# Day 7 - Step 10: Win Probability Table and Driver Analysis
# SYNTHETIC DEMONSTRATION ONLY

# Build test-set probability table
probability_table = model_df.loc[
    X_test.index,
    ["Quote_ID", "Quote_Value", "Discount_Pct",
     "Revision_Count", "Engagement_Count", "Quote_Status"]
].copy()

probability_table["Actual_Outcome"] = y_test.map({
    0: "Lost",
    1: "Won"
})

probability_table["Predicted_Win_Probability"] = y_prob
probability_table["Predicted_Outcome"] = np.where(
    y_pred == 1, "Won", "Lost"
)

probability_table["Synthetic_Data_Flag"] = True
probability_table = probability_table.sort_values(
    "Predicted_Win_Probability",
    ascending=False
)

probability_path = (
    project_dir / "data" / "synthetic"
    / "Day7_Synthetic_Quote_Win_Probability.csv"
)

probability_table.to_csv(probability_path, index=False)

# Extract standardized logistic regression coefficients.
# Positive coefficients increase the model's estimated win odds;
# negative coefficients decrease them, within this synthetic model.
logistic_model = quote_model.named_steps["model"]

driver_analysis = pd.DataFrame({
    "Feature": feature_columns,
    "Standardized_Coefficient": logistic_model.coef_[0]
})

driver_analysis["Direction"] = np.where(
    driver_analysis["Standardized_Coefficient"] > 0,
    "Positive",
    np.where(
        driver_analysis["Standardized_Coefficient"] < 0,
        "Negative",
        "Neutral"
    )
)

driver_analysis["Absolute_Coefficient"] = (
    driver_analysis["Standardized_Coefficient"].abs()
)

driver_analysis["Dataset"] = "Synthetic demonstration only"
driver_analysis = driver_analysis.sort_values(
    "Absolute_Coefficient",
    ascending=False
)

drivers_path = (
    project_dir / "data" / "synthetic"
    / "Day7_Synthetic_Quote_Driver_Analysis.csv"
)

driver_analysis.to_csv(drivers_path, index=False)

print("Probability table shape:", probability_table.shape)
print("\nHighest predicted win probabilities:")
print(probability_table[
    ["Quote_ID", "Predicted_Win_Probability",
     "Actual_Outcome", "Predicted_Outcome"]
].head(10).to_string(index=False))

print("\nDriver analysis:")
print(driver_analysis.to_string(index=False))

print("\nSaved probability table to:", probability_path)
print("Saved driver analysis to:", drivers_path)


Probability table shape: (51, 10)

Highest predicted win probabilities:
  Quote_ID  Predicted_Win_Probability Actual_Outcome Predicted_Outcome
SYN-Q-0010                   0.883889            Won               Won
SYN-Q-0001                   0.877961            Won               Won
SYN-Q-0238                   0.786470            Won               Won
SYN-Q-0012                   0.775365            Won               Won
SYN-Q-0031                   0.773401            Won               Won
SYN-Q-0240                   0.763793           Lost               Won
SYN-Q-0260                   0.717263            Won               Won
SYN-Q-0222                   0.693770            Won               Won
SYN-Q-0048                   0.675336            Won               Won
SYN-Q-0257                   0.659111           Lost               Won

Driver analysis:
         Feature  Standardized_Coefficient Direction  Absolute_Coefficient                      Dataset
    Discount_Pct         

In [21]:

# Day 7 - Step 11: Save Quote Model Evaluation Report
# SYNTHETIC DEMONSTRATION ONLY

report_path = (
    project_dir / "reports"
    / "Day7_Quote_Model_Evaluation.md"
)

report_text = f"""# Day 7 — Quotation Win Model Evaluation

## 1. Executive Summary

A baseline Logistic Regression model was developed using synthetic
quotation data for technical demonstration only.

No real quotation-level records or verified win/loss outcomes were
available in the supplied raw and cleaned prospect datasets.
Therefore, this is not a validated NexziByte business model.

## 2. Data Readiness

- Raw prospect records: 180
- Cleaned prospect records: 180
- Available quotation records: None identified
- Verified real quotation outcomes: Unavailable
- Business validation of target rules: Pending

## 3. Synthetic Dataset

- Synthetic quotations generated: 300
- Won: 118
- Lost: 137
- Pending: 45
- Records used for binary modeling: 255
- Training records: 204
- Testing records: 51

Pending quotations were excluded from binary modeling.

## 4. Features

- Quote_Value
- Discount_Pct
- Revision_Count
- Engagement_Count

The outcomes and features are synthetic. The dataset does not represent
actual NexziByte quotations or observed customer behavior.

## 5. Baseline Model

- Algorithm: Logistic Regression
- Test records: {len(y_test)}
- Accuracy: {accuracy_score(y_test, y_pred):.4f}
- Won precision: {precision_score(y_test, y_pred, zero_division=0):.4f}
- Won recall: {recall_score(y_test, y_pred, zero_division=0):.4f}
- Won F1 score: {f1_score(y_test, y_pred, zero_division=0):.4f}
- ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}

These metrics evaluate predictions on a held-out synthetic test set only.

## 6. Driver Analysis

The fitted model's standardized coefficients indicate the direction and
relative magnitude of associations learned from the synthetic dataset.
They must not be interpreted as verified business drivers.

See:
- data/synthetic/Day7_Synthetic_Quote_Driver_Analysis.csv

## 7. Win-Probability Output

The test-set probability table contains {len(probability_table)} quotations.

See:
- data/synthetic/Day7_Synthetic_Quote_Win_Probability.csv

Predicted probabilities are demonstration outputs, not real-world
quotation win probabilities.

## 8. Limitations and Risks

1. No historical quotation records were available.
2. No verified acceptance or loss labels were available.
3. Quote values, discounts, revisions, engagement histories, and loss
   reasons were not available as real quotation features.
4. Target definitions require business validation, including the
   proposed 90-day acceptance window.
5. Synthetic outcomes were generated using an illustrative probability
   mechanism; the model can learn that artificial mechanism.
6. Performance metrics cannot establish production readiness.
7. Loss-reason records are synthetic and cannot establish actual
   quotation win/loss reasons.

## 9. Recommended Next Steps

1. Request historical quotations linked to company IDs.
2. Obtain quote dates, values, discounts, revision counts, and
   engagement history where available.
3. Confirm accepted, lost, and pending status definitions with the
   business owner.
4. Confirm the proposed 90-day acceptance window and required evidence.
5. Record genuine loss reasons where available.
6. Retrain and evaluate using verified historical data.
7. Validate probability calibration and business usefulness before
   production deployment.

## 10. Conclusion

The synthetic workflow demonstrates data preparation, train/test
splitting, Logistic Regression, evaluation, probability generation,
and coefficient analysis.

A real quotation win model remains blocked until verified historical
quotation data and business-approved target definitions are available.

**Data classification: Synthetic demonstration only.**
"""

report_path.write_text(report_text, encoding="utf-8")

print("Report saved to:", report_path)
print("Report exists:", report_path.exists())
print("Report size (bytes):", report_path.stat().st_size)


Report saved to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\reports\Day7_Quote_Model_Evaluation.md
Report exists: True
Report size (bytes): 3536


In [22]:

# Day 7 - Step 12: Quotation Model Leakage Audit
# Synthetic demonstration only

leakage_checks = [
    {
        "Feature": "Quote_Value",
        "Leakage_Risk": "Low based on current schema",
        "Reason": "Quoted value can be known when the quotation is issued.",
        "Control": "Confirm the value was recorded before the prediction time."
    },
    {
        "Feature": "Discount_Pct",
        "Leakage_Risk": "Low based on current schema",
        "Reason": "Discount may be known when the quotation is issued.",
        "Control": "Use only the discount known at scoring time."
    },
    {
        "Feature": "Revision_Count",
        "Leakage_Risk": "Medium",
        "Reason": "A final revision count could include revisions after the prediction time.",
        "Control": "Use only revisions recorded up to the prediction cutoff."
    },
    {
        "Feature": "Engagement_Count",
        "Leakage_Risk": "Medium",
        "Reason": "A final engagement count could include interactions after the outcome.",
        "Control": "Count engagements only up to the prediction cutoff."
    },
    {
        "Feature": "Quote_Status",
        "Leakage_Risk": "High if used as a feature",
        "Reason": "The status directly reveals the target outcome.",
        "Control": "Exclude from model features."
    },
    {
        "Feature": "Accepted_Date",
        "Leakage_Risk": "High if used as a feature",
        "Reason": "Acceptance date may reveal that a quotation was won.",
        "Control": "Exclude from model features."
    },
    {
        "Feature": "Loss_Reason",
        "Leakage_Risk": "High if used as a feature",
        "Reason": "The recorded loss reason is outcome-related information.",
        "Control": "Use for post-outcome analysis only."
    },
    {
        "Feature": "Synthetic target generation",
        "Leakage_Risk": "Synthetic-data limitation",
        "Reason": "Outcomes were generated from an illustrative probability formula.",
        "Control": "Do not interpret model performance as real business performance."
    }
]

leakage_df = pd.DataFrame(leakage_checks)

output_path = (
    project_dir / "data" / "processed"
    / "Day7_Quote_Model_Leakage_Audit.csv"
)

leakage_df.to_csv(output_path, index=False)

print(leakage_df.to_string(index=False))
print("\nSaved to:", output_path)
print("Audit checks documented:", len(leakage_df))


                    Feature                Leakage_Risk                                                                    Reason                                                          Control
                Quote_Value Low based on current schema                   Quoted value can be known when the quotation is issued.       Confirm the value was recorded before the prediction time.
               Discount_Pct Low based on current schema                       Discount may be known when the quotation is issued.                     Use only the discount known at scoring time.
             Revision_Count                      Medium A final revision count could include revisions after the prediction time.         Use only revisions recorded up to the prediction cutoff.
           Engagement_Count                      Medium    A final engagement count could include interactions after the outcome.              Count engagements only up to the prediction cutoff.
               Quote_Stat

In [23]:

# Day 7 - Step 13: Synthetic Prediction Reason Codes
# DEMONSTRATION ONLY — NOT VERIFIED BUSINESS CAUSES

reason_table = probability_table.copy()

def generate_reason_codes(row):
    reasons = []

    if row["Discount_Pct"] >= 15:
        reasons.append("Higher discount input: positive model association")
    elif row["Discount_Pct"] < 5:
        reasons.append("Lower discount input: less positive model association")

    if row["Engagement_Count"] >= 6:
        reasons.append("Higher engagement count: positive model association")
    elif row["Engagement_Count"] <= 2:
        reasons.append("Lower engagement count: less positive model association")

    if row["Revision_Count"] >= 3:
        reasons.append("More revisions: negative model association")
    elif row["Revision_Count"] == 0:
        reasons.append("No recorded revisions: comparatively favorable model association")

    if row["Quote_Value"] >= 350000:
        reasons.append("Higher quote value: negative model association")
    elif row["Quote_Value"] < 100000:
        reasons.append("Lower quote value: comparatively favorable model association")

    return "; ".join(reasons) if reasons else (
        "No rule-based reason met the illustrative thresholds"
    )

reason_table["Reason_Codes"] = reason_table.apply(
    generate_reason_codes, axis=1
)

reason_table["Explanation_Type"] = (
    "Rule-based illustration using synthetic inputs"
)

reason_table["Synthetic_Data_Flag"] = True

reason_path = (
    project_dir / "data" / "synthetic"
    / "Day7_Synthetic_Quote_Reason_Codes.csv"
)

reason_table.to_csv(reason_path, index=False)

print("Reason-code table shape:", reason_table.shape)
print("\nSample reason codes:")
print(reason_table[
    ["Quote_ID", "Predicted_Win_Probability",
     "Predicted_Outcome", "Reason_Codes"]
].head(10).to_string(index=False))

print("\nSaved to:", reason_path)
print("All records marked synthetic:",
      bool(reason_table["Synthetic_Data_Flag"].all()))


Reason-code table shape: (51, 12)

Sample reason codes:
  Quote_ID  Predicted_Win_Probability Predicted_Outcome                                                                                                                                                                                                                           Reason_Codes
SYN-Q-0010                   0.883889               Won Higher discount input: positive model association; Higher engagement count: positive model association; No recorded revisions: comparatively favorable model association; Lower quote value: comparatively favorable model association
SYN-Q-0001                   0.877961               Won Higher discount input: positive model association; Higher engagement count: positive model association; No recorded revisions: comparatively favorable model association; Lower quote value: comparatively favorable model association
SYN-Q-0238                   0.786470               Won                            

In [24]:

# Day 7 - Step 14: Update Evaluation Report
# Synthetic demonstration only

report_path = (
    project_dir / "reports"
    / "Day7_Quote_Model_Evaluation.md"
)

additional_sections = """

## 11. Leakage Audit

The leakage audit is saved at:
- data/processed/Day7_Quote_Model_Leakage_Audit.csv

Key controls:
- Quote_Status, Accepted_Date, and Loss_Reason must not be model inputs.
- Revision and engagement features must be limited to information
  available at the prediction cutoff.
- Synthetic outcome generation limits the interpretation of all metrics.

This is a documented risk audit, not proof that every possible leakage
risk has been eliminated.

## 12. Prediction Reason Codes

The rule-based synthetic reason-code output is saved at:
- data/synthetic/Day7_Synthetic_Quote_Reason_Codes.csv

Reason codes refer to illustrative input thresholds and coefficient
directions. They are not causal explanations, verified business
drivers, or SHAP explanations.

## 13. Final Day 7 Status

Synthetic demonstration workflow: completed.

Real quotation win model: blocked pending verified historical quotation
data and business-approved target definitions.

Business validation: pending, including confirmation of the proposed
90-day acceptance window.

All synthetic outputs must remain clearly identified as demonstration
artifacts and must not be presented as actual NexziByte results.
"""

with report_path.open("a", encoding="utf-8") as report_file:
    report_file.write(additional_sections)

print("Updated report:", report_path)
print("Report exists:", report_path.exists())
print("Added leakage audit and reason-code sections.")


Updated report: D:\Nexzibyte Solution\nexzibyte-predictive-bi\reports\Day7_Quote_Model_Evaluation.md
Report exists: True
Added leakage audit and reason-code sections.


In [25]:

# Day 7 - Final Artifact Verification

required_artifacts = [
    ("Readiness audit", "data/processed/Day7_Quote_Data_Readiness_Audit.csv"),
    ("Required quotation fields", "data/processed/Day7_Required_Quotation_Fields.csv"),
    ("Target definition", "data/processed/Day7_Quote_Target_Definition.csv"),
    ("Model limitations", "data/processed/Day7_Quote_Model_Limitations.csv"),
    ("Leakage audit", "data/processed/Day7_Quote_Model_Leakage_Audit.csv"),
    ("Synthetic quotation dataset", "data/synthetic/Day7_Synthetic_Quotation_Dataset.csv"),
    ("Synthetic modeling dataset", "data/synthetic/Day7_Synthetic_Quote_Modeling_Dataset.csv"),
    ("Synthetic win probabilities", "data/synthetic/Day7_Synthetic_Quote_Win_Probability.csv"),
    ("Synthetic driver analysis", "data/synthetic/Day7_Synthetic_Quote_Driver_Analysis.csv"),
    ("Synthetic reason codes", "data/synthetic/Day7_Synthetic_Quote_Reason_Codes.csv"),
    ("Evaluation report", "reports/Day7_Quote_Model_Evaluation.md"),
]

verification_rows = []

for name, relative_path in required_artifacts:
    full_path = project_dir / relative_path
    verification_rows.append({
        "Artifact": name,
        "Path": relative_path,
        "Exists": full_path.exists()
    })

verification_df = pd.DataFrame(verification_rows)

print(verification_df.to_string(index=False))
print("\nArtifacts verified:", int(verification_df["Exists"].sum()),
      "of", len(verification_df))
print("All required artifacts exist:", bool(verification_df["Exists"].all()))


                   Artifact                                                     Path  Exists
            Readiness audit       data/processed/Day7_Quote_Data_Readiness_Audit.csv    True
  Required quotation fields        data/processed/Day7_Required_Quotation_Fields.csv    True
          Target definition          data/processed/Day7_Quote_Target_Definition.csv    True
          Model limitations          data/processed/Day7_Quote_Model_Limitations.csv    True
              Leakage audit        data/processed/Day7_Quote_Model_Leakage_Audit.csv    True
Synthetic quotation dataset      data/synthetic/Day7_Synthetic_Quotation_Dataset.csv    True
 Synthetic modeling dataset data/synthetic/Day7_Synthetic_Quote_Modeling_Dataset.csv    True
Synthetic win probabilities  data/synthetic/Day7_Synthetic_Quote_Win_Probability.csv    True
  Synthetic driver analysis  data/synthetic/Day7_Synthetic_Quote_Driver_Analysis.csv    True
     Synthetic reason codes     data/synthetic/Day7_Synthetic_Quote_Re

In [26]:
import joblib
from pathlib import Path

model_dir = Path("models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "Day7_Quote_Logistic_Model.joblib"

joblib.dump(quote_model, model_path)

print("Model saved successfully.")
print("Model path:", model_path)
print("Model type:", type(quote_model).__name__)
print("Features:", list(X.columns))

Model saved successfully.
Model path: models\Day7_Quote_Logistic_Model.joblib
Model type: Pipeline
Features: ['Quote_Value', 'Discount_Pct', 'Revision_Count', 'Engagement_Count']


In [27]:
import joblib
from pathlib import Path

model_path = Path("models/Day7_Quote_Logistic_Model.joblib")

loaded_model = joblib.load(model_path)

print("Model exists:", model_path.exists())
print("Model loaded:", loaded_model is not None)
print("Model type:", type(loaded_model).__name__)
print("Pipeline steps:", loaded_model.named_steps.keys())

Model exists: True
Model loaded: True
Model type: Pipeline
Pipeline steps: dict_keys(['scaler', 'model'])
